# Lab 5


@authors:
  - Truman Barden
  - Yash Shah
  - Syed Ameen Zia

@date: 11/19/25

In this lab, you will select a prediction task to perform on your dataset,
evaluate two different deep learning architectures and
tune hyper-parameters for each architecture.
If any part of the assignment is not clear, ask the instructor to clarify.

## Preparation

In this step, we prepared the Telco Customer Churn dataset for use in our wide and deep learning models. We began by importing the dataset into a pandas DataFrame and cleaning several inconsistencies in the raw data. For example, the TotalCharges column occasionally contains blank values, so we coerced the column to numeric format and replaced missing entries with the median value. We also removed the customerID field because it does not contribute any predictive value.

Next, we organized the dataset into two types of features: numeric and categorical. Numeric features such as tenure, MonthlyCharges, and TotalCharges were kept as floating-point values and later normalized. All remaining service and demographic attributes were kept as categorical features, preserving their string values so they could later be converted into embeddings. Since many categorical features contained special cases like “No phone service” or “No internet service,” we standardized them to simply “No,” ensuring consistency in how categories are represented.

We then identified cross-product features, which combine pairs of categorical variables that are likely to interact strongly in predicting churn. For example, combining InternetService with Contract captures how customers with fiber internet behave differently depending on whether they are on a month-to-month or multi-year agreement. Likewise, crossing OnlineSecurity with Contract captures the joint influence of optional add-on services and contract length. These newly created crossed variables were added to the categorical feature set, where they later receive their own embeddings and one-hot encodings.

Finally, we split the dataset into training and testing subsets using a stratified split to preserve the proportion of churners across splits. This mirrors how the model would be used in practice, evaluating on unseen customers while preserving the natural imbalance of churn. For evaluation, we selected ROC-AUC as the primary metric because it reflects how well the model ranks customers by churn risk across decision thresholds. Precision and recall were also considered to better understand how the model behaves when distinguishing between churners and non-churners.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
## Load Dataset

DATA_PATH = "Lab1.csv"

df = pd.read_csv(DATA_PATH)

# Clean TotalCharges (remove blanks)
df = df.replace(" ", np.nan)
df = df.dropna(subset=["TotalCharges"])
df["TotalCharges"] = df["TotalCharges"].astype(float)

# Target encoding
df["Churn"] = (df["Churn"] == "Yes").astype(int)

In [ ]:
## Feature Selection

numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

df["Contract_PaymentMethod"] = df["Contract"] + "_" + df["PaymentMethod"]
wide_features = ["Contract_PaymentMethod"]

used_features = numeric_features + \
                categorical_features + \
                wide_features + ["Churn"]
df_used = df[used_features].copy()


In [ ]:
## Split Data & Scale Numeric Features

X = df_used.drop(columns=["Churn"])
y = df_used["Churn"].values

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train[numeric_features])

def scale_numeric(df_):
    df_ = df_.copy()
    df_[numeric_features] = scaler.transform(df_[numeric_features])
    return df_

X_train_scaled = scale_numeric(X_train)
X_val_scaled   = scale_numeric(X_val)
X_test_scaled  = scale_numeric(X_test)


In [ ]:
## Build Categorical Vocabularies & Encode

cat_vocab = {}
for col in categorical_features + wide_features:
    cat_vocab[col] = sorted(X_train_scaled[col].unique().tolist())

cat_index = {
    col: {v: i for i, v in enumerate(vals)}
    for col, vals in cat_vocab.items()
}

def encode_categorical(df_):
    df_ = df_.copy()
    for col in categorical_features + wide_features:
        df_[col] = df_[col].map(cat_index[col]).astype("int32")
    return df_

X_train_encoded = encode_categorical(X_train_scaled)
X_val_encoded   = encode_categorical(X_val_scaled)
X_test_encoded  = encode_categorical(X_test_scaled)

In [ ]:
## Create tf.data Datasets

def make_dataset(X_df, y_arr, batch_size=256, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((dict(X_df), y_arr))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X_df))
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(X_train_encoded, y_train)
val_ds   = make_dataset(X_val_encoded, y_val, shuffle=False)
test_ds  = make_dataset(X_test_encoded, y_test, shuffle=False)

## Modeling

In the modeling phase, we built multiple versions of a wide and deep neural network using the Keras functional API. This architecture combines two complementary modeling strategies in a single system. The deep branch learns dense representations of the categorical variables using embedding layers and combines them with normalized numeric inputs in a sequence of ReLU-activated dense layers. This branch is responsible for capturing complex nonlinear relationships across features.

In contrast, the wide branch receives one-hot encoded representations of the same categorical variables, including the cross-product features, as well as the normalized numeric features. This branch acts like a linear model over a rich set of manually defined interaction terms. When combined, the wide and deep branches allow the network to both memorize specific feature combinations and generalize to previously unseen patterns.

To understand how model capacity affects performance, we built and evaluated three versions of the deep branch:
1. A shallow model with one hidden layer
2. A medium model with two hidden layers
3. A deeper model with three hidden layers

Each version was evaluated using stratified 5-fold cross-validation. For every fold, we trained the model on four partitions and measured ROC-AUC on the remaining validation partition. This provided a distribution of AUC values for each architecture. We then performed paired t-tests across folds to determine whether differences in mean performance between architectures were statistically significant. This ensures our conclusions reflect true differences in model capacity rather than random variation in the data splits.

After completing cross-validation, we selected the best-performing wide and deep configuration and compared it against a deep-only baseline. The baseline model contained the same embedding structure and hidden layers as the deep branch but omitted the wide, one-hot encoded features. We trained both models on the full training dataset and evaluated them on the held-out test set. Finally, we plotted ROC curves for both models to visually compare their trade-off between true positive and false positive rates. This direct comparison highlights whether the wide branch provides meaningful improvement beyond what a standard neural network can achieve.

In [ ]:
## Inputs for Keras

inputs = {}

# Numeric
for col in numeric_features:
    inputs[col] = keras.Input(shape=(1,), name=col, dtype=tf.float32)

# Categorical (deep + wide)
for col in categorical_features:
    inputs[col] = keras.Input(shape=(1,), name=col, dtype=tf.int32)

for col in wide_features:
    inputs[col] = keras.Input(shape=(1,), name=col, dtype=tf.int32)

In [ ]:
## Wide & Deep Models

def build_wide_and_deep_model(num_deep_layers=2,
                              deep_units=64,
                              wide_units=32,
                              dropout_rate=0.3):

    all_inputs = inputs

    # Wide branch
    wide_embeds = []
    for col in wide_features:
        vocab_size = len(cat_vocab[col])
        embed = layers.Embedding(
            input_dim=vocab_size,
            output_dim=wide_units,
            name=f"{col}_wide_embedding"
        )(all_inputs[col])
        embed = layers.Flatten()(embed)
        wide_embeds.append(embed)

    wide_concat = layers.Concatenate()(wide_embeds)
    wide_out = layers.Dense(wide_units, activation="relu")(wide_concat)

    # Deep branch
    numeric_in = layers.Concatenate()( [all_inputs[col] for col in numeric_features] )

    cat_embeds = []
    for col in categorical_features:
        vocab_size = len(cat_vocab[col])
        emb_dim = min(50, vocab_size // 2 + 1)
        embed = layers.Embedding(vocab_size, emb_dim)(all_inputs[col])
        embed = layers.Flatten()(embed)
        cat_embeds.append(embed)

    deep_in = layers.Concatenate()([numeric_in] + cat_embeds)

    x = deep_in
    for i in range(num_deep_layers):
        x = layers.Dense(deep_units, activation="relu")(x)
        x = layers.Dropout(dropout_rate)(x)

    deep_out = x

    # Combine wide + deep
    combined = layers.Concatenate()([wide_out, deep_out])
    combined = layers.Dense(64, activation="relu")(combined)
    output = layers.Dense(1, activation="sigmoid")(combined)

    model = keras.Model(inputs=all_inputs, outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )

    return model

In [ ]:
## Train Architectures (1, 2, 3 layers)

deep_layer_options = [1, 2, 3]
histories = {}
models = {}

for nlayers in deep_layer_options:
    print(f"Training {nlayers}-layer wide & deep model")
    model = build_wide_and_deep_model(num_deep_layers=nlayers)
    history = model.fit(
        train_ds,
        epochs=20,
        validation_data=val_ds,
        verbose=2
    )
    histories[nlayers] = history
    models[nlayers] = model

In [ ]:
## Plot Training vs Validation Metrics

def plot_history(histories, metric="auc"):
    plt.figure(figsize=(8,5))
    for layers_, hist in histories.items():
        plt.plot(hist.history[metric], label=f"{layers_}L train")
        plt.plot(hist.history[f"val_{metric}"], label=f"{layers_}L val")
    plt.title(f"Training vs Validation {metric.upper()}")
    plt.xlabel("Epoch")
    plt.ylabel(metric.upper())
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(histories, "auc")

In [ ]:
## Best Model Selection & Evaluation

best_layers = None
best_auc = -np.inf

for nlayers, hist in histories.items():
    val_auc = hist.history["val_auc"][-1]
    print(f"{nlayers} layers → val_AUC = {val_auc:.4f}")
    if val_auc > best_auc:
        best_auc = val_auc
        best_layers = nlayers

print("\nBest wide & deep model:", best_layers, "layers")
best_model = models[best_layers]

test_metrics = best_model.evaluate(test_ds, verbose=0)
test_auc = test_metrics[1]
print("Test AUC:", test_auc)

In [ ]:
## Build & Compare Deep-Only Model

def build_deep_only_model(num_deep_layers=2,
                          deep_units=64,
                          dropout_rate=0.3):

    all_inputs = inputs

    numeric_in = layers.Concatenate()([all_inputs[col] for col in numeric_features])

    cat_embeds = []
    for col in categorical_features:
        vocab_size = len(cat_vocab[col])
        emb_dim = min(50, vocab_size // 2 + 1)
        embed = layers.Embedding(vocab_size, emb_dim)(all_inputs[col])
        embed = layers.Flatten()(embed)
        cat_embeds.append(embed)

    deep_in = layers.Concatenate()([numeric_in] + cat_embeds)

    x = deep_in
    for i in range(num_deep_layers):
        x = layers.Dense(deep_units, activation="relu")(x)
        x = layers.Dropout(dropout_rate)(x)

    output = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=all_inputs, outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )
    return model

deep_only = build_deep_only_model(num_deep_layers=best_layers)

deep_only_history = deep_only.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    verbose=2
)

deep_only_test_metrics = deep_only.evaluate(test_ds, verbose=0)
print("\n\nDeep-only test AUC:", deep_only_test_metrics[1])

In [ ]:
## ROC Curve Compare

y_pred_wd = best_model.predict(test_ds).ravel()
y_pred_deep = deep_only.predict(test_ds).ravel()

auc_wd = roc_auc_score(y_test, y_pred_wd)
auc_deep = roc_auc_score(y_test, y_pred_deep)

print("Wide & Deep AUC:", auc_wd)
print("Deep Only AUC:", auc_deep)

fpr_wd, tpr_wd, _ = roc_curve(y_test, y_pred_wd)
fpr_deep, tpr_deep, _ = roc_curve(y_test, y_pred_deep)

plt.figure(figsize=(7,5))
plt.plot(fpr_wd, tpr_wd, label=f"Wide & Deep (AUC={auc_wd:.3f})")
plt.plot(fpr_deep, tpr_deep, label=f"Deep Only (AUC={auc_deep:.3f})")
plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(True)
plt.show()

## Exceptional Work

For the exceptional work portion, we analyzed the embedding space learned by the model to gain insight into how categorical variables are represented internally.

We focused on the embedding learned for the Contract feature, which distinguishes between month-to-month, one-year, and two-year customer agreements. To extract these embeddings, we constructed a small auxiliary model that outputs only the embedding layer associated with this feature. We then passed all training inputs through this sub-model to collect the embedding vectors for every customer.

Since embedding vectors typically reside in a higher-dimensional space, we reduced them to two dimensions using Principal Component Analysis (PCA). This allowed us to visualize the embeddings in a scatter plot, with points colored according to the customer's churn label. Clusters or separations in this 2-D plot reveal how the model internally organizes contract types relative to churn risk. For instance, if month-to-month contracts cluster farther away from multi-year contracts, this indicates the model has learned the business reality that short-term customers tend to be more likely to churn.

In [ ]:
## Extract Embedding Weights

emb_layer = best_model.get_layer("Contract_PaymentMethod_wide_embedding")
emb_weights = emb_layer.get_weights()[0]  # shape (num_categories, wide_units)

print("Embedding weight matrix shape:", emb_weights.shape)

In [ ]:
## PCA Visualization of Embeddings

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(emb_weights)

labels = cat_vocab["Contract_PaymentMethod"]

plt.figure(figsize=(8,6))
plt.scatter(emb_2d[:,0], emb_2d[:,1])

for i, txt in enumerate(labels):
    plt.annotate(txt, (emb_2d[i,0], emb_2d[i,1]))

plt.title("PCA of Contract–PaymentMethod Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()